In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

import imageio.v2 as imageio
from IPython.display import display, Video, Markdown
from io import BytesIO

import sys
sys.path.extend(["../../"])

from utils import categorical_crossentropy_loss_function, grad_categorical_crossentropy_loss_wrt_softmax_model,computation_graph_softmax,create_computation_graph_linear


In [ ]:
display(Markdown(open("../../_macros.md").read()))

# Multiclass Classification

In the same way we have studied problems in which we perform regression over 1 output and over $C$ outputs, we can naturally extend the problem of two-class classification (which we framed as a regression to the interval $[0,1]$) to $ C$- class classification. Here we perform regression to the $C-1$ dimensional probability simplex: https://en.wikipedia.org/wiki/Simplex.

As an example, our $ C$- class classification problem will model classification of three animals: dog, cat, and hamster. We now have a label $t\in\{1,\dots,C\}$. Let's build a toy dataset with three classes instead of two. Here we will directly use two features: weight ($x_1$) and height ($x_2$), since it provides a nicer visualization. The labels are represented by integer values $t\in\{1,\dots,C\}$ for implementation purposes. However, for theoretical description and derivations, we use the one-hot representation (see below)

In [ ]:
x_data = np.array([
    [0.0, 0.0], [0.5, 0.5], [1.0, 0.0], [0.5, 1.0],       # hamster
    [3.0, 0.0], [3.5, 0.5], [4.0, 0.0], [3.5, -0.5],      # cat
    [1.5, 3.0], [2.0, 3.5], [2.5, 3.0], [2.0, 4.5],       # dog
])
labels = np.array([0]*4 + [1]*4 + [2]*4)
C = 3

class_names = ["hamster", "cat", "dog"]
markers = ["o", "s", "^"]

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
for c in range(C):
    idx = labels == c
    ax.plot(x_data[idx, 0], x_data[idx, 1], markers[c], color=f"C{c}", markersize=9,
             markeredgecolor="k", label=class_names[c])
ax.set_xlabel("weight ($x_1$)")
ax.set_ylabel("height ($x_2$)")
ax.legend()


## A model for this data.

Here, as always, we can break the modelling procedure into the model and the link function. We will use a linear model, although, as always, we can replace $\xvec$ with some linear basis functions. For the same reasons as in the binary case, we are not really interested in label assignment but rather in a probabilistic assignment over the $C$ classes. Thus, rather than using a link function that maps the input to the class label assignment, we will be using a link function that outputs a $ C$- dimensional probabilistic vector $\pvec$. The link function shall be constructed in a way such that the elements of $\pvec$ are non-negative, with values between 0 and 1, and the sum of the elements is $1$. Otherwise, $\pvec$ is not a probabilistic vector.

For this, we can use the Softmax link function, given by:

$$
p_c = \text{softmax}(\zvec)_c = \frac{e^{z_c}}{\sum^C_{j=1}e^{z_j}}, \qquad c=1,\dots,C
$$

Unlike the sigmoid, which acts on one number at a time, softmax genuinely mixes all $C$ logits together to produce each $p_c$, and that is exactly what guarantees $\sum_c p_c=1$ by construction. Now each class $c$ gets its own linear model, $z_c=\xvect\wvec_c+b_c$, so instead of one weight vector we now have a $D\times C$ weight matrix $\Wmat=(\wvec_1,\dots,\wvec_C)$ and a length-$C$ bias vector $\bvec$.

Thus, the overall computational graph is given by:

$$
\begin{split}
\zvect = \xvect\Wmat+\bvect \\
\pvect=\text{softmax}(\zvect)
\end{split}
$$

For $N$ training points we have (introducing $\bvec$ through a vector of ones into $\Xmat$):

$$
\begin{split}
\Zmat = \Xmat\Wmat \\
\Pmat=\text{softmax}(\Zmat)
\end{split}
$$

where the softmax acts row-wise. Note that the softmax function, compared to the rest of the functions we have seen, does not act element-wise.

Two important aspects of the softmax:

* It is order-preserving, meaning that the highest logit has the highest probability value.
* It is shift invariant, meaning that if we add a constant to $\zvec$, we get exactly the same probability vector.

There are two consequences regarding this observation. First, for a purely classification task, ie, to decide the winning class, there is no need to compute the softmax. On the other hand, there are infinite models that give rise to the same loss function. Why? Because adding the same vector $\Vvec\in\mathbb{R}^D$ to every column $\wvec_c$ of $\Wmat$, together with the same scalar $\beta$ added to every entry of $\bvec$, shifts every logit $z_c=\xvect\wvec_c+b_c$ by the identical amount $\xvect\Vvec+\beta$ regardless of $c$, so $\pvec$ (and thus the loss) is completely unchanged. Thus, even if the loss landscape might be convex, the minimum is not a single point but a $(D+1)$-dimensional affine subspace of $(\Wmat,\bvec)$: the translate, by any one minimizer, of the $(D+1)$-dimensional linear subspace of shifts $\{(\Vvec\onevect,\beta\onevect):\Vvec\in\mathbb{R}^D,\beta\in\mathbb{R}\}$. We will visualize this later.

### Computing the softmax stably

A practical note before moving on. Computing $p_c=e^{z_c}/\sum_j e^{z_j}$ literally as written can overflow: if any logit is even moderately large (say $z_j=1000$), $e^{z_j}$ is already `inf` in floating point, and `inf/inf` evaluates to `nan`.

The fix reuses exactly the shift-invariance property from above: subtracting the same constant from every logit in $\zvec$ leaves $\pvec$ unchanged, so subtract the row-wise maximum $m=\max_j z_j$ before exponentiating:

$$
p_c = \frac{e^{z_c}}{\sum_j e^{z_j}} = \frac{e^{z_c-m}}{\sum_j e^{z_j-m}}, \qquad m=\max_j z_j.
$$

Now every exponent $z_c-m\leq 0$, so every $e^{z_c-m}\in(0,1]$: nothing overflows, and the largest term in the denominator is exactly $1$, so the sum can never underflow to $0$ either. This is precisely what `activation_function_softmax` implements in `utils.py`.


### $C=2$ is exactly logistic regression

Before going further, let's make sure this new machinery is at least consistent with what we already had. For $C=2$, dividing numerator and denominator by $e^{z_1}$:

$$
p_1 = \frac{e^{z_1}}{e^{z_1}+e^{z_2}} = \frac{1}{1+e^{-(z_1-z_2)}} 
$$

So softmax with $2$ classes is exactly the sigmoid, evaluated at the difference of the two logits, and $p_2=1-p_1$ exactly as before. Multiclass classification is a genuine generalization of what we already built, not a different model.


### Display possible linear models.


Let's visualize some of these models. Following the 2-dimensional input plot we used for the two-binary case, rather than printing functions (since I do not know a good way to plot the softmax similar to how we plot the sigmoid), we directly plot the regions assigned to each class, with varying degrees of color intensity to denote the levels of probability assigned to each class. This plot only shows the value of the winning probability, but not the values assigned to other classes.

For similar reasons as  before, linear models will learn linear decision thresholds. The proof is left as an exercise **Exercise**. 

The decision rule is now "predict the class with the highest probability", $\hat t=\arg\max_c p_c$. Since softmax's $p_c$ only depends on the *differences* between logits, the boundary between any two classes $c,j$ is where $z_c=z_j$, a straight line (or hyperplane); with $C>2$ classes, these lines meet and carve the input space into $C$ convex polygonal regions, exactly the multiclass generalization of the single line/plane we got for $C=2$. Let's see this for some randomly chosen candidate models.


In [ ]:
np.random.seed(3)

x1_grid, x2_grid = np.meshgrid(np.linspace(-2, 6, 300), np.linspace(-3, 6, 300))
grid_flat = np.stack([x1_grid.ravel(), x2_grid.ravel()], axis=1)

cmap = ListedColormap(["C0", "C1", "C2"])

num_models_to_show = 10
class_cmaps = ["Blues", "Oranges", "Greens"]  # matches the C0/C1/C2 colors used for the scatter markers

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for i in range(num_models_to_show):

    ## create a candidate model
    w, b = create_computation_graph_linear(2, C)
    b = b * 0.3  # one bias per class; computation_graph_softmax accepts (K,1) or (1,K)
    w = w * 0.3

    ## probabilities over the grid
    p_grid = computation_graph_softmax(grid_flat, w, b)
    winner = np.argmax(p_grid, axis=1).reshape(x1_grid.shape)
    p_winner = np.max(p_grid, axis=1).reshape(x1_grid.shape)

    ax.cla()

    ## shade each class's region by the winning probability, one colormap per class
    for c in range(C):
        p_masked = np.where(winner == c, p_winner, np.nan)
        cf = ax.contourf(x1_grid, x2_grid, p_masked, levels=np.linspace(1/C, 1, 10),
                          cmap=class_cmaps[c], alpha=0.8)
        if i == 0:
            cbar = fig.colorbar(cf, ax=ax)
            cbar.set_label(f"P({class_names[c]}) when winning")

    ax.contour(x1_grid, x2_grid, winner, levels=[0.5, 1.5], colors="k", linewidths=1.5)

    for c in range(C):
        idx = labels == c
        ax.plot(x_data[idx, 0], x_data[idx, 1], markers[c], color=f"C{c}", markersize=9,
                markeredgecolor="k", label=class_names[c])
    ax.set_xlabel("weight ($x_1$)")
    ax.set_ylabel("height ($x_2$)")
    ax.set_title(f"Model {i+1}/{num_models_to_show} (untrained, random)")
    ax.legend(loc="upper left")

    ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
    ## save images for later display
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    buf.seek(0)
    frame = imageio.imread(buf)
    writer.append_data(frame)

writer.close()
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

## Loss function

For the loss functions, we have two options. Both are generalizations of the Brier Score and the Binary Cross-Entropy to $C$ classes. Here we will only focus on the generalization of the Binary Cross-Entropy to $C$ classes. In fact, the generalization of the Brier Score to $C$ classes results in the loss we used for multi-output regression when there is independence between outputs. 

Encode the label as a one-hot vector $\tvec\in\{0,1\}^C$ ($t_c=1$ for the true class, $0$ otherwise). The natural generalization of BCE to $C$ classes 
is known as the categorical cross-entropy. It can be associated with maximum likelihood for the Categorical distribution **Exercise**:

$$
\begin{split}
L_{\text{CCE}} &= -\sum^N_{n=1}\sum^C_{c=1} t_c^n\log p_c^n\\
&= -\tr{\Tmatt\log\Pmat}
\end{split}
$$

where $\log\Pmat$ denotes the elementwise (not matrix) logarithm of $\Pmat$, which reduces exactly to $L_{\text{BCE}}$ when $C=2$: writing out the inner sum's two terms with $t_2^n=1-t_1^n$ and $p_2^n=1-p_1^n$ gives back $-\sum_n t_1^n\log p_1^n+(1-t_1^n)\log(1-p_1^n)$, precisely the loss we already know. Like BCE, it is a proper scoring rule.

As with the binary case, the categorical cross-entropy results in a convex loss **Exercise** when combined with a linear model; it also penalizes totally wrong predictions with an infinite loss. In general, all the properties we see in the binary case hold here.

Let's evaluate these losses within the random functions created before.:



In [ ]:
np.random.seed(3)

x1_grid, x2_grid = np.meshgrid(np.linspace(-2, 6, 300), np.linspace(-3, 6, 300))
grid_flat = np.stack([x1_grid.ravel(), x2_grid.ravel()], axis=1)

cmap = ListedColormap(["C0", "C1", "C2"])

num_models_to_show = 10
class_cmaps = ["Blues", "Oranges", "Greens"]  # matches the C0/C1/C2 colors used for the scatter markers

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))
plt.subplots_adjust(wspace=0.3)

loss_per_model = []
for i in range(num_models_to_show):

    ## create a candidate model
    w, b = create_computation_graph_linear(2, C)
    b = b * 0.3  # one bias per class; computation_graph_softmax accepts (K,1) or (1,K)
    w = w * 0.3

    ## probabilities over the grid and over the data
    p_grid = computation_graph_softmax(grid_flat, w, b)
    winner = np.argmax(p_grid, axis=1).reshape(x1_grid.shape)
    p_winner = np.max(p_grid, axis=1).reshape(x1_grid.shape)

    p_data = computation_graph_softmax(x_data, w, b)
    loss_per_model.append(categorical_crossentropy_loss_function(labels, p_data).sum())

    ax1.cla()
    ax2.cla()

    ## shade each class's region by the winning probability, one colormap per class
    for c in range(C):
        p_masked = np.where(winner == c, p_winner, np.nan)
        cf = ax1.contourf(x1_grid, x2_grid, p_masked, levels=np.linspace(1/C, 1, 10),
                           cmap=class_cmaps[c], alpha=0.8)
        if i == 0:
            cbar = fig.colorbar(cf, ax=ax1)
            cbar.set_label(f"P({class_names[c]}) when winning")

    ax1.contour(x1_grid, x2_grid, winner, levels=[0.5, 1.5], colors="k", linewidths=1.5)

    for c in range(C):
        idx = labels == c
        ax1.plot(x_data[idx, 0], x_data[idx, 1], markers[c], color=f"C{c}", markersize=9,
                 markeredgecolor="k", label=class_names[c])
    ax1.set_xlabel("weight ($x_1$)")
    ax1.set_ylabel("height ($x_2$)")
    ax1.set_title(f"Model {i+1}/{num_models_to_show} (untrained, random)")
    ax1.legend(loc="upper left")

    ## loss incurred by each model so far
    ax2.plot(np.arange(1, len(loss_per_model) + 1), loss_per_model, 'o-', color='k')
    ax2.set_xlim([0, num_models_to_show + 1])
    ax2.set_xlabel("i-th model")
    ax2.set_ylabel("loss")
    ax2.set_title("Categorical cross-entropy loss")

    ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
    ## save images for later display
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    buf.seek(0)
    frame = imageio.imread(buf)
    writer.append_data(frame)

writer.close()
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

### Convexity of the loss function and non-identifiable statistical model


This section shows how the categorical cross-entropy loss endowed with a linear model results in a convex loss function. It also shows the non-identifiability of this statistical model.

Non-identifiable statistical models are those statistical models in which a set of different parameters give rise to the same statistical model. Mixture models are another example.

To be able to illustrate this loss function, we consider a $C=2$ class problem with inputs $x\in \mathbb{R}$, and no bias. In this way, we have two parameters $w_1$ and $w_2$, each one in charge of each class. The loss function can be written down through the following composition.

$$
\begin{split}
z_1 &= w_1\cdot x\\
z_2 &= w_2\cdot x\\
\pvec &= \text{softmax}([z_1,z_2]^T)\\
L &= -\tvect\log\pvec = -t_1\log p_1 - t_2\log p_2
\end{split}
$$

Recall from the claim above that adding the same vector $\Vvec\in\mathbb{R}^D$ to every column of $\Wmat$, together with the same scalar $\beta$ added to every entry of $\bvec$, leaves $\pvec$ (and thus the loss) unchanged: a $(D+1)$-dimensional invariant subspace in general. Here $D=1$ and there is no bias, so only the $\Vvec$ part of that argument applies, and since $D=1$, $\Vvec\in\mathbb{R}^1$ is itself just a scalar $v$. Let's check directly what shifting $w_1,w_2$ by the same $v$ does to the model, starting from the composition above: take $w_1'=w_1+v$ and $w_2'=w_2+v$. The new logits are

$$
z_1' = w_1'x = (w_1+v)x = w_1x+vx = z_1+vx, \qquad z_2' = w_2'x = w_2x+vx = z_2+vx,
$$

so both logits shift by the exact same amount $vx$, and hence $\pvec$ and the loss are exactly unchanged, for every $v\in\mathbb{R}$. So if $(w_1,w_2)$ achieves the minimum loss, then $(w_1,w_2)+\alpha(1,1)$ achieves that very same minimum loss for every $\alpha\in\mathbb{R}$: an entire line of equally-good minimizers, not an isolated point. This gives rise to the line

$$
r(\alpha) = (w_1,w_2) + \alpha(1,1), \qquad \alpha\in\mathbb{R}.
$$ 

Let's plot the loss function alongside some of these lines. **Exercise:** Show that this function is convex. **Exercise:** How can we modify the model to make it identifiable? I mean so that there is a unique maximum/minimum in the loss function.

In [ ]:
np.random.seed(0)

## toy 1D, 2-class dataset (no bias): a few points per class, NOT linearly separable
## (one point per class sits on the "wrong" side of x=0, so no (w1,w2) can drive the loss to 0)
x_2c = np.array([-2.5, -2.0, -1.5, 0.3, -0.3, 1.5, 2.0, 2.5]).reshape(-1, 1)
labels_2c = np.array([0, 0, 0, 0, 1, 1, 1, 1])

## grid of (w1, w2) values
w1_grid = np.linspace(-10, 5, 200)
w2_grid = np.linspace(-5, 10, 200)
W1, W2 = np.meshgrid(w1_grid, w2_grid)

## Vectorized loss over the whole grid, using computation_graph_softmax and
## categorical_crossentropy_loss_function exactly as defined in utils.py.
## Trick: z1=w1*x, z2=w2*x is bilinear in (w,x), so we can swap their roles and feed the
## (w1,w2) grid as the "data" batch, using diag(x_n,x_n) as the "model" for each data point.
## This turns the huge grid loop (G*G iterations) into a tiny loop over the N=8 data points.
def grid_loss(w1_pts, w2_pts, x, labels):
    grid_pairs = np.stack([w1_pts.ravel(), w2_pts.ravel()], axis=1)  # (M,2)
    M = grid_pairs.shape[0]
    L_flat = np.zeros(M)
    for xn, tn in zip(x[:, 0], labels):
        Wn = np.array([[xn, 0.0], [0.0, xn]])  # z1=w1*xn, z2=w2*xn, for every grid point at once
        bn = np.zeros((2, 1))
        p_n = computation_graph_softmax(grid_pairs, Wn, bn)
        L_flat += categorical_crossentropy_loss_function(np.full(M, tn), p_n).ravel()
    return L_flat.reshape(w1_pts.shape)

L_surf = grid_loss(W1, W2, x_2c, labels_2c)

## the non-identifiable direction, parametrized directly as (w1_0+v, w2_0+v), the same
## "add the same v to both" shift used in the theory above -- starting from the minimum
## found on the grid and sweeping v as far as it stays inside the grid on both axes
i_min, j_min = np.unravel_index(np.argmin(L_surf), L_surf.shape)
w1_0, w2_0 = W1[i_min, j_min], W2[i_min, j_min]
v_lo = max(w1_grid.min() - w1_0, w2_grid.min() - w2_0)
v_hi = min(w1_grid.max() - w1_0, w2_grid.max() - w2_0)
v = np.linspace(v_lo, v_hi, 100)
w1_line = w1_0 + v
w2_line = w2_0 + v
L_line = grid_loss(w1_line, w2_line, x_2c, labels_2c)

## a few other non-identifiable lines (different w2-w1 offsets), to show that each one is
## individually flat but only the red one (through the actual minimum) sits at the lowest value
other_d_list = [-3.0, -1.0, 1.0, 3.5]
other_lines = []
for d_other in other_d_list:
    w1_0_other, w2_0_other = 0.0, d_other
    v_lo_other = max(w1_grid.min() - w1_0_other, w2_grid.min() - w2_0_other)
    v_hi_other = min(w1_grid.max() - w1_0_other, w2_grid.max() - w2_0_other)
    v_other = np.linspace(v_lo_other, v_hi_other, 100)
    w1_o = w1_0_other + v_other
    w2_o = w2_0_other + v_other
    L_o = grid_loss(w1_o, w2_o, x_2c, labels_2c)
    other_lines.append((w1_o, w2_o, L_o))

fig = plt.figure(figsize=(13, 5))
ax1 = fig.add_subplot(1, 2, 1)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

cf = ax1.contourf(W1, W2, L_surf, levels=30, cmap="viridis")
for w1_o, w2_o, L_o in other_lines:
    ax1.plot(w1_o, w2_o, color="green", linewidth=2)
ax1.plot(w1_line, w2_line, color="red", linewidth=2.5)
ax1.set_xlabel("$w_1$")
ax1.set_ylabel("$w_2$")
ax1.set_title("CCE loss contour")
fig.colorbar(cf, ax=ax1)

ax2.plot_surface(W1, W2, L_surf, cmap="viridis", linewidth=0, antialiased=True, alpha=0.9)
## small z-offset so the lines render above the surface instead of fighting its depth-sorting
z_offset = 0.02 * (L_surf.max() - L_surf.min())
for w1_o, w2_o, L_o in other_lines:
    ax2.plot(w1_o, w2_o, L_o + z_offset, color="green", linewidth=2.5)
ax2.plot(w1_line, w2_line, L_line + z_offset, color="red", linewidth=3)
ax2.view_init(elev=10, azim=45)  # tweak elev/azim to rotate the view
ax2.set_xlabel("$w_1$")
ax2.set_ylabel("$w_2$")
ax2.set_zlabel("loss")
ax2.set_title("CCE loss surface")

plt.tight_layout()

## Optimization

Model optimization, in this case, relies on gradient descent. It is the most efficient way of optimizing this model as far as I know. When I say gradient descent, I mean second-order methods as well, whenever $C$ is not so big that matrix inversions for second-order optimization algorithms are reasonable.

Here we will see that the softmax in combination with the cross-entropy loss yields very well-behaved gradients in terms of computation. 

We start, as always, by expressing the loss as a function composition:

$$
\begin{split}
\Zmat &= \Xmat\Wmat \\
\Pmat &= \text{softmax}(\Zmat)\\
\Lmat &= \log\Pmat \quad \text{element-wise}\\
 l &= -\tr{\Tmatt\Lmat}
\end{split}
$$

with $\Xmat\in\mathbb{R}^{N\times D}$ and $\Wmat\in\mathbb{R}^{D\times C}$ (so $\Zmat,\Pmat,\Lmat\in\mathbb{R}^{N\times C}$, $l\in\mathbb{R}$), the Jacobian of each step is:

$$
\begin{align*}
J_\Zmat &= \Imat\otimes\Xmat && \in \mathbb{R}^{NC\times DC}\\
J_\Pmat &=  K_{NC}\bra{\diag\pare{\vvec\Pmatt} -   \diag\pare{\vvec(\Pmatt)}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec(\Pmatt)} }K_{CN} && \in \mathbb{R}^{NC\times NC}\\
J_\Lmat &= \diag\pare{\vvec(1/\Pmat)} && \in \mathbb{R}^{NC\times NC}\\
J_l &= -\vvec(\Tmat)^T && \in \mathbb{R}^{1\times NC}
\end{align*}
$$

some of these Jacobians have already been derived across the book; others can be found here https://arxiv.org/pdf/2506.23996. The one on the Softmax is below.  By the chain rule,

$$
\begin{split}
J_{\vvec\Wmat} l &= J_l J_\Lmat J_\Pmat J_\Zmat \in \mathbb{R}^{1\times DC}\\
&=-\vvec(\Tmat)^T\diag\pare{\vvec(1/\Pmat)} K_{NC}\bra{\diag\pare{\vvec\Pmatt} -   \diag\pare{\vvec(\Pmatt)}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec(\Pmatt)} }K_{CN}\Imat\otimes\Xmat\\
&= \pareT{\vvec\bra{\Pmat-\Tmat}}\pare{\Imat\otimes\Xmat}
\end{split}
$$

The simplification can also be found below. As you see, the Jacobian of the softmax activation, in contrast to the other problems we have seen such as regression or binary classification, is non-diagonal. **Exercise** Why?


Note that this Jacobian is in vectorized form, which means the canonical form is:

$$
\dd l = J_{\vvec\Wmat} \dd \vvec \Wmat
$$

The associated gradient will be given by its transpose, which means (using standard identities from the Kronecker product):

$$
\nabla_{\vvec{\Wmat}} l = \pare{\Imat\otimes\Xmatt}\vvec\bra{\Pmat-\Tmat}
$$

Note that this is the gradient wrt a vectorized form of $\Wmat$. I think this is the first time in this book this appears in the way I am going to write it because in multioutput regression we were using OLS. There, setting the gradient to zero, we finally undo the vectorization to express something in matrix form. Here, note that if we want to apply gradient descent directly with this gradient, we would need to keep our matrix as a vector, and reshape it to a matrix each time we want to use it. That is not very useful. It is preferable to use the gradient in the unvectorized form. Note that applying the standard identity $\vvec\pare{\Amat\Xmat\Bmat}=\pare{\Bmatt \otimes \Amat}\vvec \Xmat$ we have:

$$
\begin{split}
\nabla_{\vvec{\Wmat}} l &= \pare{\Imat\otimes\Xmatt}\vvec\bra{\Pmat-\Tmat}\\
&=\vvec\pare{\Xmatt\bra{\Pmat-\Tmat}}
\end{split}
$$

If we unvec on both sides, we get:

$$
\begin{split}
\nabla_{\Wmat} l &= \Xmatt\bra{\Pmat-\Tmat}
\end{split}
$$

which is the gradient that uses the code function of this notebook. Well, the code function separates the computation of $\Wmat$ from $\bvec$. But that is easy to extrapolate from here. We just need to change the composition and use:

$$
\begin{split}
\Zmat &= \Xmat\Wmat + \onevec_N \bvect 
\end{split}
$$

$\onevec_N \bvect $ is a very elegant way to write down the broadcasting operator that sums a column vector over each single row. As always, figuring out algebraic forms allows us to systematically derive Jacobians and gradients.

Now, the gradient wrt $\Wmat$ is exactly the one we have derived by removing the column of ones from $\Xmat$. For the gradient over $\bvec$, we start from the Jacobian of $\Zmat$ wrt $\bvec$. Here, the canonical form (matrix-valued, vector-argument function) is:

$$
\begin{split}
\dd \vvec \Amat = J \dd \xvec
\end{split}
$$

Let's work this out. Starting from:

$$
\begin{split}
\Zmat &= \Xmat\Wmat + \onevec_N \bvect\\
\vvec \Zmat &= \vvec\bra{\Xmat\Wmat + \onevec_N \bvect}\\ 
\dd \vvec \Zmat &= \dd \vvec\bra{\Xmat\Wmat + \onevec_N \bvect}\\ 
\dd \vvec \Zmat &=  \vvec\bra{\onevec_N \dd \bvect\Imat_C}\\
\dd \vvec \Zmat &=  \bra{\Imat_C \otimes \onevec_N} \dd \vvec \bvect\\
\dd \vvec \Zmat &=  \bra{\Imat_C \otimes \onevec_N} \dd\bvec\\
\end{split}
$$

since $\vvec \bvect = \vvec \bvec = \bvec$. So, in this case the Jacobian wrt the bias is:

$$
\begin{split}
J_{\bvec} l &=  \pareT{\vvec\bra{\Pmat-\Tmat}}\bra{\Imat_C \otimes \onevec_N}
\end{split}
$$

Here, the bias is a vector directly. The transpose gives the gradient:

$$
\begin{split}
\nabla_{\bvec} l &=  \bra{\Imat_C \otimes \onevect_N}\vvec\bra{\Pmat-\Tmat}
\end{split}
$$

A bit of algebra using the previous identity:

$$
\begin{split}
\nabla_{\bvec} l &=  \bra{\Imat_C \otimes \onevect_N}\vvec\bra{\Pmat-\Tmat}\\
&= \vvec\pare{\onevect_N\bra{\Pmat-\Tmat}\Imat_C }\\
&= \vvec\pare{\onevect_N\bra{\Pmat-\Tmat}}\\
&= \braT{\Pmat-\Tmat}\onevec_N
\end{split}
$$

The last step notes that inside vec there is a product between a row vector and a matrix, which gives a row vector. We apply $\vvec \bvect = \vvec \bvec = \bvec$ again. The resulting gradient is a column vector as expected. This gradient is basically summing up the rows of $\Pmat-\Tmat$. In other words, it is collapsing the $N$ dimension.

### Jacobian of the softmax


#### Formulation 1

For a single vector $\zvec\in\mathbb{R}^C$ with $\pvec=\text{softmax}(\zvec)$, the Jacobian is

$$
J_{\pvec\zvec} = \diag(\pvec) - \pvec\pvect
$$

This result is the compact algebra form of writting the results derived here: https://eli.thegreenplace.net/2016/the-softmax-function-and-its-derivative/. However, for our usual function composition we write:

$$
\begin{split}
\zvect &= \xvect\Wmat \\
\pvect &= \text{softmax}(\zvect)\\
\dots
\end{split}
$$

Obtaining the Jacobian here, might look tricky. Why? Because we need to tranpose on both sides from $\pvect = \text{softmax}(\zvect)$, and since the softmax is a non-linear operator we might think there is no rule that allow us to workout how the transpose affect the softmax. However, by inspection, it can be easily checked that. $\braT{\text{softmax}(\zvect)}=\text{softmax}(\zvec)$. Thus, the Jacboian remains the same $\diag(\pvec) - \pvec\pvect$.


However, this trick cannot be applied when both entriy and output  are  matrices, ie when $N\neq1$ and we have more than one training point. In this case, in which the output is a matrix with the softmax computed rowise, we can derive a compact expression by inspection. First note that the row-wise softmax give raise to the matrix $\Pmat$. Suppose with have two training points N and 3 classes, then:

$$
\Pmat = \begin{pmatrix} p_{11} & p_{12} & p_{13} \\ p_{21} & p_{22} & p_{23} \end{pmatrix}
$$

where we use $p_{nc}$ where $n$ indicates training point and $c$ class. So each row is the sofmax for each training point. We keep using the row-wise convention where each row represents a point. The Jacobian is a block diagonal matrix which each block taking the value $(\diag(\pvec) - \pvec\pvect)_n$ for the corresponding $n$ point:

$$
J = \left[\begin{array}{ccc|ccc}
p_{11}(1-p_{11}) & -p_{11}p_{12} & -p_{11}p_{13} & 0 & 0 & 0\\
-p_{12}p_{11} & p_{12}(1-p_{12}) & -p_{12}p_{13} & 0 & 0 & 0\\
-p_{13}p_{11} & -p_{13}p_{12} & p_{13}(1-p_{13}) & 0 & 0 & 0\\
\hline
0 & 0 & 0 & p_{21}(1-p_{21}) & -p_{21}p_{22} & -p_{21}p_{23}\\
0 & 0 & 0 & -p_{22}p_{21} & p_{22}(1-p_{22}) & -p_{22}p_{23}\\
0 & 0 & 0 & -p_{23}p_{21} & -p_{23}p_{22} & p_{23}(1-p_{23})
\end{array}\right]
$$

Okay, how do we actually compute this in a compact form. Well, very easy. Note that we could start by extending $\braT{\diag(\pvec) - \pvec\pvect}$ to deal with matrices. This is very easy, we just need to replace $\pvec$ by the vector associated to a matrix which is $\vvec\Pmat$. However, by convention $\vvec\Pmat$ creates a vector by stacking columns, and we need to follow a row-wise order. Thus we need $\vvec(\Pmatt)=K_{CN}\vvec(\Pmat)$, with $K_{CN}\in\mathbb{R}^{NC\times NC}$ being the Commutation matrix (see https://tminka.github.io/papers/matrix/minka-matrix.pdf). This gives raise to:

$$
\braT{\diag\pare{K_{CN}\vvec\Pmat} - \pare{K_{CN}\vvec\Pmat}\braT{K_{CN}\vvec\Pmat}}
$$

With $K_{CN}\vvec(\Pmat)=(p_{11},p_{12},p_{13},p_{21},p_{22},p_{23})^T$. This also gives raise to a symmetric matrix:

$$
\left[\begin{array}{ccc|ccc}
p_{11}(1-p_{11}) & -p_{11}p_{12} & -p_{11}p_{13} & -p_{11}p_{21} & -p_{11}p_{22} & -p_{11}p_{23}\\
-p_{12}p_{11} & p_{12}(1-p_{12}) & -p_{12}p_{13} & -p_{12}p_{21} & -p_{12}p_{22} & -p_{12}p_{23}\\
-p_{13}p_{11} & -p_{13}p_{12} & p_{13}(1-p_{13}) & -p_{13}p_{21} & -p_{13}p_{22} & -p_{13}p_{23}\\
\hline
-p_{21}p_{11} & -p_{21}p_{12} & -p_{21}p_{13} & p_{21}(1-p_{21}) & -p_{21}p_{22} & -p_{21}p_{23}\\
-p_{22}p_{11} & -p_{22}p_{12} & -p_{22}p_{13} & -p_{22}p_{21} & p_{22}(1-p_{22}) & -p_{22}p_{23}\\
-p_{23}p_{11} & -p_{23}p_{12} & -p_{23}p_{13} & -p_{23}p_{21} & -p_{23}p_{22} & p_{23}(1-p_{23})
\end{array}\right]
$$

So the only step is to multiply by a matrix of ones in the block diagonal and 0 in the rest. This can be done through $\Imat_{N}\otimes\onevec_C\onevect_C$, where $\Imat_N$ is the $N\times N$ identity matrix and $\onevec_C$ is a vector of ones with dimension $C$. This gives:

$$
\Imat_{2}\otimes\onevec_3\onevect_3 = \left[\begin{array}{ccc|ccc}
1 & 1 & 1 & 0 & 0 & 0\\
1 & 1 & 1 & 0 & 0 & 0\\
1 & 1 & 1 & 0 & 0 & 0\\
\hline
0 & 0 & 0 & 1 & 1 & 1\\
0 & 0 & 0 & 1 & 1 & 1\\
0 & 0 & 0 & 1 & 1 & 1
\end{array}\right]
$$ 

The Hadamard product of this matrix with $\diag\pare{K_{CN}\vvec\Pmat} - \pare{K_{CN}\vvec\Pmat}\braT{K_{CN}\vvec\Pmat}$ is the Jacobian, thus:

$$
J = \bra{\diag\pare{K_{CN}\vvec\Pmat} - \pare{K_{CN}\vvec\Pmat}\pareT{K_{CN}\vvec\Pmat}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C}
$$

However, note that this is the Jacobian whenever   $\vvec\Pmatt$ against $\vvec \Zmatt$. Note that the first row in the matrix is the derivative of $p_{11}$ against $z_{11},z_{12},z_{13},z_{21},z_{22},z_{23}$, which are exactly  the vec operator over the transposed matrix. However the canonical form here says:

$$
\begin{split}
\dd \vvec \Pmat = J \dd \vvec \Zmat
\end{split}
$$

Thus, starting from:

$$
\begin{split}
\dd \vvec \Pmatt = J' \dd \vvec \Zmatt
\end{split}
$$

we can use the Conmutation matriz as follows:

$$
\begin{split}
& \dd \vvec \Pmatt = J' \dd \vvec \Zmatt\\
& K_{CN}\dd \vvec \Pmat = J' K_{CN} \dd \vvec \Zmat\\
& \dd \vvec \Pmat = K_{NC} J' K_{CN} \dd \vvec \Zmat\\
\end{split}
$$

which is in canonical form. Thus the Jacobian of this step is given by:

$$
\begin{split}
J = K_{NC}\left[\bra{\diag\pare{K_{CN}\vvec\Pmat} - \pare{K_{CN}\vvec\Pmat}\pareT{K_{CN}\vvec\Pmat}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C}\right]K_{CN}
\end{split}
$$

##### Simplying this Jacobian

First of all we replace the identity  $\vvec(\Pmatt)=K_{CN}\vvec(\Pmat)$, to start removing matrices from the identity.

$$
\begin{split}
J = K_{NC}\left[\bra{\diag\pare{\vvec\Pmatt} - \pare{\vvec\Pmatt}\pareT{\vvec\Pmatt}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C}\right]K_{CN}
\end{split}
$$

Now using the fact that Hadamard distributes over subtraction we can write:

$$
\begin{split}
&K_{NC}\bra{\diag\pare{\vvec\Pmatt}-\vvec(\Pmatt)\pareT{\vvec\Pmatt}}\circ\pare{\Imat_N\otimes\onevec_C\onevect_C}K_{CN} = \\
&K_{NC}\bra{\diag\pare{\vvec\Pmatt}\circ\pare{\Imat_N\otimes\onevec_C\onevect_C}} -\bra{\vvec(\Pmatt)\pareT{\vvec\Pmatt}\circ\pare{\Imat_N\otimes\onevec_C\onevect_C}}K_{CN}
\end{split}
$$

Let's work with the first element of the sum. Note that $\Imat_N\otimes\onevec_C\onevect_C$ has ones in the diagonal, thus the hadamart product of a diagonal matrix with this matrix is just the diagonal matrix. This gives:

$$
\begin{split}
K_{NC}\bra{\diag\pare{\vvec\Pmatt} -\bra{\vvec(\Pmatt)\pareT{\vvec\Pmatt}\circ\pare{\Imat_N\otimes\onevec_C\onevect_C}}}K_{CN}
\end{split}
$$

Continuing with the second term of the sum, and using eq 77 in Minka and the fact that $\Amat\circ\Bmat = \Bmat \circ\Amat$.


$$
\begin{split}
K_{NC}\bra{\diag\pare{\vvec\Pmatt} -   \diag\pare{\vvec(\Pmatt)}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec(\Pmatt)} }K_{CN}
\end{split}
$$

##### Simplfying the whole Jacobian.

One of the well known facts about Cross Entropy Loss plus Softmax is that it yields a very well-behaved and stable operation.  Looking at the Jacobian: 

$$
-\vvec(\Tmat)^T\diag\pare{\vvec(1/\Pmat)} K_{NC}\bra{\diag\pare{\vvec\Pmatt} -   \diag\pare{\vvec(\Pmatt)}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec(\Pmatt)} }K_{CN}\Imat\otimes\Xmat
$$

where we had divisions over $p$. This can be unstable, since when the model converges $p$ gets values around $1$ for the true class and $0$ for the others, and dividing by zero leads to very high values that can result in numerical unstable computations (overflow/underflow).

Let's see how the three operations: sum, cross entropy loss and softmax lead to a very well behave gradient. We can simplify the gradient as follows. This is important because automatic differentiation software does not make the funciton composition I have done here, but directly the composition: sum+crossentropy+softmax so that the gradient is directly this nice expression. Start from:

$$
\begin{split}
J_{\vvec\Wmat} l &= J_l J_\Lmat J_\Pmat \\
&=-\vvec(\Tmat)^T\diag\pare{\vvec(1/\Pmat)} K_{NC}\bra{\diag\pare{\vvec\Pmatt} -   \diag\pare{\vvec(\Pmatt)}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec(\Pmatt)} }K_{CN}\\
\end{split}
$$

First, multiplying a vector by a diagonal matrix scales each entry by its corresponding entry element in the matrix, so by inspection:

$$
-\vvec(\Tmat)^T\diag\pare{\vvec(1/\Pmat)} = -\vvec(\Tmat/\Pmat)^T
$$

This gives:

$$
J_{\vvec\Wmat} l = -\vvec(\Tmat/\Pmat)^T K_{NC}\bra{\diag\pare{\vvec\Pmatt} -   \diag\pare{\vvec(\Pmatt)}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec(\Pmatt)} }K_{CN}
$$

Now, transposing the well known identity on both sides $K_{CN}\vvec(\Amat)=\vvec(\Amatt)$ directly gives $\vvec(\Amat)^TK_{NC}=\vvec(\Amatt)^T$. With $\Amat=\Tmat/\Pmat$:

$$
-\vvec(\Tmat/\Pmat)^T K_{NC} = -\pareT{\vvec\pareT{\Tmat/\Pmat}}
$$

and since transposing an elementwise division just transposes each matrix separately, $\pareT{\Tmat/\Pmat}=\Tmatt/\Pmatt$, so this is simply:

$$
-\pareT{\vvec\pareT{\Tmat/\Pmat}} = -\vvec\pare{\Tmatt/\Pmatt}^T
$$

This gives:

$$
J_{\vvec\Wmat} l = -\vvec\pare{\Tmatt/\Pmatt}^T\bra{\diag\pare{\vvec\Pmatt} -   \diag\pare{\vvec(\Pmatt)}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec(\Pmatt)} }K_{CN}
$$

Using now the identity: $\vvec\pare{\Amat \circ \Bmat}=\diag(\vvec \Amat)\vvec\Bmat$, then transpose to yield $\pareT{\vvec\pare{\Amat \circ \Bmat}}=\pareT{\vvec\Bmat}\pareT{\diag(\vvec \Amat)}=\pareT{\vvec\Bmat}\diag(\vvec \Amat)$. Then:

$$
\begin{split}
&-\vvec\pareT{\Tmatt/\Pmatt}\diag\pare{\vvec\Pmatt}=\\
&\vvec\pare{\Pmatt\circ\Tmatt/\Pmatt}=\vvec\Tmatt
\end{split}
$$

This yields:

$$
\begin{split}
J_{\vvec\Wmat} l = \bra{-\vvec\Tmatt + \vvec\pare{\Tmatt}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec(\Pmatt)} }K_{CN}
\end{split}
$$

We can now use $\pareT{\vvec\pare{\Amat \Xmat \Bmat}} = \pareT{\pare{\Bmatt \otimes \Amat} \vvec \Xmat}= \pareT{\vvec\Xmat}\pareT{\Bmatt \otimes \Amat}$, and using the fact that $\pare{\Imat_N\otimes\onevec_C\onevect_C}$ and $\onevec_C\onevect_C$ are symmetric.

$$
\begin{split}
&\pareT{\vvec\Tmatt}\pare{\Imat_N\otimes\onevec_C\onevect_C}\diag\pare{\vvec\Pmatt}=\\
&\pareT{\vvec\pare{\onevec_C\onevect_C \Tmatt \Imat_N}}\diag\pare{\vvec\Pmatt}
\end{split}
$$

Using again $\pareT{\vvec\pare{\Amat \circ \Bmat}}=\pareT{\vvec\Bmat}\pareT{\diag(\vvec \Amat)}=\pareT{\vvec\Bmat}\diag(\vvec \Amat)$, this can be simplified further:

$$
\begin{split}
&\pareT{\vvec\pare{\onevec_C\onevect_C \Tmatt \Imat_N}}\diag\pare{\vvec\Pmatt}=\\
&\pareT{\vvec\pare{\pare{\onevec_C\onevect_N \Imat_N}\circ \Pmatt}}=\\
&\pareT{\vvec\pare{\pare{\onevec_C\onevect_N}\circ \Pmatt}}=\\
&\pareT{\vvec\Pmatt}
\end{split}
$$

Since: $\Tmatt$ is a matrix with columns containing one $1$ and the rest elements of zero. This means that $\onevect_C \Tmatt=\onevect_N$ gives a row vector of ones. $\onevec_C\onevect_N$ is  C times N matrix of ones. Hadamart product with $\Pmatt$ (C times N matrix) is just $\Pmatt$. Putting together:


$$
\begin{split}
J_{\vvec\Wmat} l &= \braT{-\vvec\Tmatt + \vvec\Pmatt }K_{CN}\\
&= \braT{\vvec\bra{\Pmatt-\Tmatt}}K_{CN}\\
\end{split}
$$

Using again  $\vvec(\Amat)^TK_{NC}=\vvec(\Amatt)^T$ and the fact that $K_{NC}K_{CN}=\Imat$:

$$
J_{\vvec\Wmat} l = \pareT{\vvec\bra{\Pmat-\Tmat}}
$$

This the Jacobian corresponding to this composition: sum, plus softmax plus cross entropy:

$$
\begin{split}
\Pmat &= \text{softmax}(\Zmat)\\
\Lmat &= \log\Pmat\\
 l &= -\tr{\Tmatt\Lmat}
\end{split}
$$

Thus, the whole Jacobian is:

$$
J_{\vvec\Wmat} l = \pareT{\vvec\bra{\Pmat-\Tmat}}\Imat\otimes\Xmat
$$

#### Formulation 2

This formulation comes from an interesting fact Claude pointed out when I was making him write down the $N=3$, $C=2$ Jacobian for the softmax, in an aim to derive a vectorized form. Obviously I know what the Jacobian looks like (it is actually in the webpage I linked) is just a matter of writting takes ages and Claudes does it in a minute. Once I finished obtaining the expression $J = \bra{\diag\pare{\vvec\Pmatt} - \pare{\vvec\Pmatt}\pareT{\vvec\Pmatt}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C}$ Claude pointed me to the fact that this Jacobian was the one resulting from the function $\Pmatt = \text{softmax}(\Zmatt)$. In fact is true, and is the reason why commutation matrices appear. I wrote down the differentials, workout the canonical forms and realize that I need to multiply by commutation on both sides.

This is one of the examples where AI makes ourselves quicker. Not because I do not realize about this. Look, this would have been the old pipeline when AI was not avaible:

* I would have taken this Jacobian and try and reach the expression $\pareT{\vvec\pare{\Pmat-\Tmat}}$, because I already know this derivative yields to this expression since it is a very well known fact from the log loss and softmax.
* I woudl have seen there is no way to obtain the expression. Probably, It would have taken a bunch of time through either writting small examples as with $N=2$ $C=3$ to realize expression did not let to $\Pmat-\Tmat$.
* Then, either I would let this as a TODO or get the gradient I know from many of the forums where it is written (https://shivammehta25.github.io/posts/deriving-categorical-cross-entropy-and-softmax/), code both gradients up and realized they are different. You can observe how this path is much easier than mine, but mine can be generalized to tensors of arbitrary shape so I wanted to make this natural step.
* Finally I would have eventually come to the same observation as Claude, or just think I am stupid.

*A note:* Obviously working out things in vectorized form is much better than using partial derivatives to get vectorized forms by inspection. Throughout all the book I haven't figured out if many of the expressions I obtained can be simplified. Probably it can, but that is something one focus when going for an implementation. And actually I usually care about efficient per-step vector Jacobian computations when this comes into matter. However science usually go the other way, sometimes. For instance the well known fact from the log loss and softmax is that it yields a Jacobian which is simply obtained by $\Pmat - \Tmat$. This is readily seen by inspection (as done here (https://shivammehta25.github.io/posts/deriving-categorical-cross-entropy-and-softmax/)). Here, I wanted to go one step further and generalize this result through matrix calculus, which is a more general perspective. So as with smetimes science is, I start with a small result and generalize it.

*Some thoughts on the use of AI I found while writting this chapter:* I usually find that AI not always makes the best decision towards writting mathematical derivations. In this chapter, I derived the vectorized form of the Jacobian (except pointing to commutation matrix). AI stucked with expression involving the canonical matrix base and I do not remember what more things. I just wanted a clean expression as the one I derived. When it came to simplification I prompt claude to make it but did not like what I observed. For instance it did not realize applying *$\pareT{\vvec\pare{\Amat \circ \Bmat}}=\pareT{\vvec\Bmat}\pareT{\diag(\vvec \Amat)}=\pareT{\vvec\Bmat}\diag(\vvec \Amat)$* was the fastest option in that part of the derivation. On the other side, he pointed to equation 77 in Minka (not directly but as a standard relation between outer product and Hadamart product. In fact Minka applies it in a very standard way). It was actually that step in the simplification and pointing to standard identity $\vvec\pare{\Amat\Xmat\Bmat}$ relation to kronecker product which lead to a very simple simplification. In my original derivation I directly targeted the simplification of the Jacobian of the loss wrt the input to the softmax directly, since that is what directly obtaines the expression $t-p$. With this focus Claude was able to realize how applying standard algebra identities lead to the simplification, after I started simplifying the initial steps. So I shall say that Claude is like a very nice "google search" when you point it in the direct direction. I sometimes provide the best path, it sometimes provides the best path. It is a matter of how fresh do we have the identities in our mind. So Claude was important in this book both for part of the simplification and realizing that I miss the commutation matrix the expression I derived for the Jacobian of the Softmax. It is very nice to see how it makes me avoid very time consuming steps of working out some products on code or paper when I do not realize about the identities or I just get some result wrong, such as missing the commutation matrix. On the other side, sometimes you think it is so good that it miss things that are trivial. For instance in chapter 1 "section considering the special case  $f:\mathbb{R}^D\rightarrow\mathbb{R}^C$, $N=1$" I realized that instead of using a row vector it was expressing everything through a matrix of one row. While it is correct, it might result in a more efficient Jacobian if we workout directly the things using vectores. Here realizing that the canonical form uses column vectors and that we can go through different steps as I did was something Claude was unable to do correctly. My main takeaway is that claude is helping me writting latex math code which is usually very time consuming. Good team work!.

The important thing about this observation from claude is what motivate me to add the special case in chapter 1 "section considering the special case  $f:\mathbb{R}^D\rightarrow\mathbb{R}^C$, $N=1$". We will see that this same trick allow us to get read off conmutation matrix, and eventually our problem would not have appeared. In other words, sometimes using other function compositions make things easier. Suppose we change our function composition to:

$$
\begin{split}
\Zmatt &= \Wmatt\Xmatt \\
\Pmatt &= \text{softmax}(\Zmatt)\\
\Lmatt &= \log\Pmatt\\
 l &= -\tr{\Tmat\Lmatt}
\end{split}
$$

This function composition is equivalent. But what is important from here? Well the expression $J = \bra{\diag\pare{K_{CN}\vvec\Pmat} - \pare{K_{CN}\vvec\Pmat}\pareT{K_{CN}\vvec\Pmat}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C}$ is now the actual Jacobian from the softmax step. Note that here we want the canonical form of $\dd\vvec{\Pmatt} = J \dd\vvec{\Zmatt}$ so when the inputs are actually this transposed matrix. The rest of the Jacobians are:

$$
\begin{align*}
J_\Zmatt &= \bra{\Xmat\otimes\Imat} K_{CD}\\
J_\Pmatt &=  \bra{\diag\pare{\vvec\Pmatt} - \pare{\vvec\Pmatt}\pareT{\vvec\Pmatt}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C}\\
J_\Lmatt &= \diag\pare{\vvec(1/\Pmatt)} \\
J_l &= -\vvec(\Tmatt)^T 
\end{align*}
$$

Here, the commutation matrix appears in the first expression. I would not have made this error because the canonical form of this differential can be obtained using well known identities applied in a systematic way, and once you reach $\dd \vvec\Wmatt$ you know that commutation matrix is the last step towards canonical form. Is not like softmax where I derive it by inspection. 

By the chain rule,

$$
\begin{split}
J_{\vvec\Wmat} l &= J_l J_\Lmatt J_\Pmatt J_\Zmatt \in \mathbb{R}^{1\times DC}\\
&= -\vvec(\Tmatt)^T\diag\pare{\vvec(1/\Pmatt)} \bra{\diag\pare{\vvec\Pmatt} - \pare{\vvec\Pmatt}\pareT{\vvec\Pmatt}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C} \bra{\Xmat\otimes\Imat} K_{CD}
\end{split}
$$

Note that how the role of the commutation matrix is directly absorved into the Jacobians. As we see $\Tmatt$ and $\Pmatt$ already appeared transposed. Let's go and simplify:

Again using the fact that a row vector multiplied by a diagonal matrix is a row vector with per elements multiplications we have:

$$
\begin{split}
J_{\vvec\Wmat} l &= J_l J_\Lmatt J_\Pmatt J_\Zmatt \in \mathbb{R}^{1\times DC}\\
&= -\vvec(\Tmatt)^T\diag\pare{\vvec(1/\Pmatt)} \bra{\diag\pare{\vvec\Pmatt} - \pare{\vvec\Pmatt}\pareT{\vvec\Pmatt}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C} \bra{\Xmat\otimes\Imat} K_{CD}\\
&=-\vvec\pareT{\Tmatt/\Pmatt} \bra{\diag\pare{\vvec\Pmatt} - \pare{\vvec\Pmatt}\pareT{\vvec\Pmatt}} \circ \pare{\Imat_{N}\otimes\onevec_C\onevect_C} \bra{\Xmat\otimes\Imat} K_{CD}\\
\end{split}
$$

Now the rest of operations remains the same yielding:

$$
\begin{split}
 \pareT{\vvec\bra{\Pmatt-\Tmatt}} \bra{\Xmat\otimes\Imat} K_{CD}
\end{split}
$$

Using again  $\vvec(\Amat)^TK_{NC}=\vvec(\Amatt)^T$:

$$
\begin{split}
\pareT{\vvec\bra{\Pmat-\Tmat}} K_{NC} \bra{\Xmat\otimes\Imat} K_{CD}
\end{split}
$$

Note that it is exactly the same Jacobian as we obtained before:

$$
\pareT{\vvec\bra{\Pmat-\Tmat}}\pare{\Imat\otimes\Xmat}
$$

Since: $\pare{\Imat\otimes\Xmat}= K_{NC} \bra{\Xmat\otimes\Imat} K_{CD}$. This is not a standard identity as far as I know but can be checked through code. If I gone this path I would have never had this problem. This is usually the reason why I make derivations through different paths to check they are correct, and is in fact what I did when learning about differentials, as you can see here: https://arxiv.org/pdf/2506.23996

### Running gradient descent

As always, let's now implementa and run gradient descent.


In [ ]:
np.random.seed(0)

x1_grid, x2_grid = np.meshgrid(np.linspace(-2, 6, 300), np.linspace(-3, 6, 300))
grid_flat = np.stack([x1_grid.ravel(), x2_grid.ravel()], axis=1)
class_cmaps = ["Blues", "Oranges", "Greens"]  # matches the C0/C1/C2 colors used for the scatter markers

w, b = create_computation_graph_linear(2, C, mean=0, std=0)  # deterministic zero init, b is (C,1)
lr = 0.1
epochs = 100
frame_every = 1  

video_filename = "/tmp/aux.mp4"
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=10, codec="libx264")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
fig.subplots_adjust(wspace=0.3)

loss_history = []
for e in range(epochs):
    p = computation_graph_softmax(x_data, w, b)
    loss_history.append(categorical_crossentropy_loss_function(labels, p).sum())
    grad_w, grad_b = grad_categorical_crossentropy_loss_wrt_softmax_model(x_data, labels, w, b)

    if e % frame_every == 0 or e == epochs - 1:
        p_grid = computation_graph_softmax(grid_flat, w, b)
        winner = np.argmax(p_grid, axis=1).reshape(x1_grid.shape)
        p_winner = np.max(p_grid, axis=1).reshape(x1_grid.shape)

        ax1.cla()
        ax2.cla()

        ## shade each class's region by the winning probability, one colormap per class
        for c in range(C):
            p_masked = np.where(winner == c, p_winner, np.nan)
            cf = ax1.contourf(x1_grid, x2_grid, p_masked, levels=np.linspace(1/C, 1, 10),
                               cmap=class_cmaps[c], alpha=0.8)
            if e == 0:
                cbar = fig.colorbar(cf, ax=ax1)
                cbar.set_label(f"P({class_names[c]}) when winning")

        ax1.contour(x1_grid, x2_grid, winner, levels=[0.5, 1.5], colors="k", linewidths=1.5)
        for c in range(C):
            idx = labels == c
            ax1.plot(x_data[idx, 0], x_data[idx, 1], markers[c], color=f"C{c}", markersize=9,
                     markeredgecolor="k", label=class_names[c])
        ax1.set_xlabel("5weight ($x_1$)")
        ax1.set_ylabel("height ($x_2$)")
        ax1.set_title(f"iteration {e}")
        ax1.legend(loc="upper left")

        ax2.plot(loss_history, color="C0")
        ax2.set_xlim([0, epochs])
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Loss")
        ax2.set_title("CCE loss")

        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
        buf.seek(0)
        frame = imageio.imread(buf)
        writer.append_data(frame)

    w = w - lr * grad_w
    b = b - lr * grad_b

writer.close()
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

## Measuring performance

Investigate yourself performance metrics for the multiclass case: accuracy, multiclass confusion matrix, macro/micro-averaged precision, recall, F1.

## TODO

* For a $D=1$ input, plot each class probability $p_c(x)$ as a curve (like the sigmoid plot).
* Regularized multiclass logistic regression.
* Linear basis function
* Newton's method: check if that gives the same IRLS method.
* Evaluation metrics beyond accuracy: multiclass confusion matrix, macro/micro-averaged precision, recall, F1.
* Label smoothing: softening the one-hot $\tvec$ itself, a different (and complementary) way of regularizing the same loss. Robust loss function
* Ordinal classification, when the $C$ classes have a natural order (unlike hamster/cat/dog) and treating them as unordered throws information away.
